<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/11_marca_de_agua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Marca de agua de Google Maps (issue #7)

Las imágenes de Jamaica (`kingston`, `kingston2` y `geo`) son capturas de Google Maps, y algunos recortes traen la
marca de agua al pie. Las otras fuentes no. El test de Kingston también va a ser de Google Maps, así que un modelo
podría aprender "marca de agua → Jamaica → sin paneles" en vez de mirar los paneles.

Este notebook es el **primer paso** del issue: ubicar en qué imágenes aparece y dónde, y ver si se asocia con la
clase. No entrena nada ni usa Drive (`entorno.preparar(usar_drive=False)`).

La marca tiene tres partes (detalle en `pdi_tp/marca_de_agua.py`):

- **Barra de atribución**: franja gris oscura con "Imagery ©2025 Airbus, CNES / Airbus, Maxar Technologies".
- **Logo "Google"**: letras blancas con contorno oscuro, abajo a la izquierda.
- **"© 2025 Google" casi transparente**, repetido sobre toda la imagen. Es muy tenue y no se detecta.

El detector es a mano (umbrales calibrados mirando las imágenes): revisar las grillas de la sección 3.

In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdi_tp import datos, entorno, marca_de_agua, split
rutas = entorno.preparar(usar_drive=False)  # no entrena: no monta Drive ni crea carpetas

## 1. En qué imágenes aparece

Se corre el detector sobre las 900 imágenes. Si aparece alguna fuera de Jamaica, es un falso positivo del detector.

In [ ]:
division = split.cargar_split(rutas)
conjunto = {p.name: nombre for nombre, lista in division.items() for p in lista}
filas = []
for imagen in datos.listar_imagenes(rutas):
    deteccion = marca_de_agua.detectar(datos.leer_imagen(imagen))
    filas.append({"archivo": imagen.name, "fuente": split.fuente(imagen.stem), "jamaica": split.es_jamaica(imagen.name),
                  "split": conjunto.get(imagen.name), "con_paneles": int(datos.tiene_paneles(datos.label_de(rutas, imagen))),
                  **deteccion})
tabla = pd.DataFrame(filas)

resumen = tabla.groupby("fuente").agg(imagenes=("archivo", "size"), con_marca=("marca", "sum"),
                                      barra=("barra", "sum"), logo=("logo", "sum"))
resumen["% con marca"] = (100 * resumen["con_marca"] / resumen["imagenes"]).round(1)
print("Fuera de Jamaica con marca (falsos positivos):", int(tabla.loc[~tabla["jamaica"], "marca"].sum()))
resumen

## 2. ¿Se asocia con la clase?

Si la marca aparece mucho más en imágenes sin paneles, el modelo la puede usar como atajo. Hay dos comparaciones:

- **En todo el dataset**: la marca solo aparece en Jamaica, y Jamaica tiene muchas más imágenes sin paneles que el
  resto. Así que, aunque la marca no tenga nada que ver con los paneles, "marca → sin paneles" funciona en train.
- **Solo dentro de Jamaica**: ahí se ve si la marca agrega información además de la fuente.

In [ ]:
def proporciones(datos_, titulo):
    cruce = pd.crosstab(datos_["marca"].map({True: "con marca", False: "sin marca"}),
                        datos_["con_paneles"].map({1: "con paneles", 0: "sin paneles"}))
    cruce["% sin paneles"] = (100 * cruce.get("sin paneles", 0) / cruce.sum(axis=1)).round(1)
    print(titulo)
    display(cruce)


proporciones(tabla, "Todo el dataset (900)")
proporciones(tabla[tabla["jamaica"]], "Solo Jamaica")
for nombre in ("train", "val"):
    proporciones(tabla[tabla["jamaica"] & (tabla["split"] == nombre)], f"Jamaica, {nombre}")

print("Clase por fuente dentro de Jamaica (otro posible atajo, sin relación con la marca):")
pd.crosstab(tabla.loc[tabla["jamaica"], "fuente"], tabla.loc[tabla["jamaica"], "con_paneles"]
            .map({1: "con paneles", 0: "sin paneles"}))

## 3. Revisión a ojo

Todas las detecciones, y una muestra de imágenes de Jamaica **sin** detección para buscar las que se le escaparon
al detector (sobre todo recortes con un pedacito de barra o de logo).

In [ ]:
def grilla(nombres, titulo, columnas=8):
    nombres = list(nombres)
    if not nombres:
        print(titulo, "— ninguna")
        return
    filas_ = -(-len(nombres) // columnas)
    fig, axs = plt.subplots(filas_, columnas, figsize=(2 * columnas, 2.15 * filas_))
    for ax in np.ravel(axs):
        ax.set_axis_off()
    for ax, nombre in zip(np.ravel(axs), nombres):
        fila = tabla.set_index("archivo").loc[nombre]
        ax.imshow(datos.leer_imagen(rutas.imagenes / nombre))
        partes = [p for p in ("barra", "logo") if fila[p]]
        ax.set_title(f"{fila['fuente']} · {'+'.join(partes) or '—'} · {'con' if fila['con_paneles'] else 'sin'}",
                     fontsize=7)
    fig.suptitle(titulo, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


grilla(tabla.loc[tabla["marca"], "archivo"], "Con marca detectada")
sin_deteccion = tabla[tabla["jamaica"] & ~tabla["marca"]]
grilla(sin_deteccion.sample(min(40, len(sin_deteccion)), random_state=0)["archivo"],
       "Jamaica sin marca detectada (muestra)")

## 4. Qué zona ocupa

Para cada fila de píxeles, la diferencia entre las imágenes con marca y las de Jamaica sin marca en la fracción de
píxeles grises (sin color), que es lo que tiene la barra. Donde la diferencia se dispara empieza la marca. La línea
punteada es `marca_de_agua.FILA_INICIO_MARCA`, la fila desde la que habría que tapar.

In [ ]:
def fraccion_gris(nombres):
    total = np.zeros(256)
    for nombre in nombres:
        imagen = datos.leer_imagen(rutas.imagenes / nombre).astype(float)
        total += (np.ptp(imagen, axis=2) < 12).mean(1)
    return total / max(len(nombres), 1)


jamaica = tabla[tabla["jamaica"]]
diferencia = fraccion_gris(jamaica.loc[jamaica["marca"], "archivo"]) - fraccion_gris(jamaica.loc[~jamaica["marca"], "archivo"])
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(range(256), diferencia)
ax.axvline(marca_de_agua.FILA_INICIO_MARCA, color="gray", ls="--")
ax.set(xlabel="fila de la imagen (0 = arriba)", ylabel="diferencia de píxeles grises",
       title="Con marca − sin marca, Jamaica")
plt.show()
print(f"Tapar desde la fila {marca_de_agua.FILA_INICIO_MARCA}: "
      f"{256 - marca_de_agua.FILA_INICIO_MARCA} filas, {100 * (256 - marca_de_agua.FILA_INICIO_MARCA) / 256:.0f} % de la imagen.")

## Próximos pasos

1. **Elegir cómo neutralizarla**, con lo que mostraron las secciones 1 y 2:
   - **Tapar la franja de abajo** (`FILA_INICIO_MARCA` en adelante) en **todas** las imágenes: train, val y test,
     de todas las fuentes. Es simple y saca la barra y el logo, pero pierde esa franja también donde hay paneles.
   - **Pegar marcas falsas** en imágenes de las otras fuentes como augmentation (`pdi_tp/aumentacion.py`), para que
     la marca deje de indicar "Jamaica". No pierde información, pero es más trabajo.
2. **Medir el efecto en Jamaica**, sobre todo el acierto en las imágenes con paneles: entrenar el mismo modelo con
   y sin el cambio, con la receta común, y compararlos con `10_comparacion_final`.
3. La parte casi transparente ("© 2025 Google" repetido) no se puede tapar así. Si el modelo la usara, la
   augmentation con marcas falsas también la cubriría, imitándola.